# 08 - EDA para Machine Learning: Itens de Pedido + Rastreamento
**Squad 1 — Data Quality em Tempo Real | ML em Streaming**
**Responsável:** Riuler José Silvério
**Problema:** P1 · Anomalias em pedidos e itens · **Modelo A:** Isolation Forest (não supervisionado clássico)
**Comparação:** Modelo B (Autoencoder) de Guilherme & Carlos, sobre `ecommerce_pedidos`

### Objetivo
Verificar se os dados das Silvers estão adequados para o modelo, conhecer as tabelas e levantar insights que orientam a Engenharia de Features (notebook `09`). Este notebook **só lê**: não grava nada.

### Perguntas que a EDA responde
1. Volume, período, nulos e duplicatas: a base é suficiente para treinar?
2. Qual é o **rótulo fraco** (itens reprovados pelas regras da etapa 1) e quanto ele representa?
3. Como se distribuem preço, quantidade e desconto? Onde estão os valores impossíveis e os fora do padrão?
4. Qual é o tamanho dos pedidos (itens e valor)?
5. O rastreamento tem informação suficiente para "horas entre eventos"?
6. As features candidatas separam os itens rotulados dos não rotulados?
7. A base é estável entre lotes? Isso define a divisão treino × teste.

In [0]:
# Credenciais via Secret Scope (nunca no código)
adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

_conta = f"{adls_storage_account}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{_conta}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{_conta}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{_conta}": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{_conta}": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{_conta}": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

base_squad1 = f"abfss://squad1@{_conta}"
caminhos = {
    "silver_itens": f"{base_squad1}/dupla1/silver/ecommerce_itens_pedido",
    "silver_rastreamento": f"{base_squad1}/dupla1/silver/ecommerce_rastreamento_entregas",
    "silver_produtos": f"{base_squad1}/silver/ecommerce_produtos",          # outra dupla, só leitura
    "features_itens": f"{base_squad1}/dupla1/ml/features_itens_pedido",     # saída do notebook 09
}
print("Credenciais e caminhos prontos.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Regras por linha da Silver de itens (True = passou, False = falhou, null = não avaliada)
REGRAS_ITENS = [
    "tecnica_1_pk_valida", "tecnica_2_fk_pedido_valido", "tecnica_3_fk_sku_valido",
    "tecnica_4_quantidade_valida", "tecnica_5_preco_valido",
    "negocio_6_desconto_nao_maior_preco", "negocio_8_desconto_nao_negativo",
    "negocio_9_desconto_dentro_limite",
]
ORDEM_STATUS = {"em separacao": 1, "coletado": 2, "em transito": 3, "saiu para entrega": 4, "entregue": 5}

def com_lote(df):
    """Extrai o micro-lote (AAAA/MM/DD/HHMMSS) do caminho de origem."""
    lote = F.regexp_extract("bronze_source_file", r"(\d{4}/\d{2}/\d{2}/\d{6})", 1)
    return (df.withColumn("lote", lote)
              .withColumn("dt_lote", F.to_timestamp("lote", "yyyy/MM/dd/HHmmss"))
              .withColumn("data_lote", F.to_date("dt_lote")))

def sem_duplicatas(df, chave):
    """A Silver é append: fica a versão mais recente de cada chave."""
    w = Window.partitionBy(chave).orderBy(F.col("silver_processed_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def status_normalizado(col):
    """'EM TRANSITO', 'em_transito' e 'Em transito' viram 'em transito'."""
    return F.trim(F.lower(F.regexp_replace(col, "_", " ")))

### Etapa 1 — Leitura das Silvers

Lê as duas Silvers da dupla 1 e o catálogo de produtos (preço de lista por SKU). Com cerca de 14 mil itens e 3 mil eventos, cabe em pandas para os gráficos.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

AZUL, VERMELHO, AMARELO, CINZA = "#1F4E8C", "#C0392B", "#F2B705", "#7F8C8D"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})

ler = lambda p: spark.read.format("delta").options(**adls_options).load(p)

df_itens_s = com_lote(sem_duplicatas(ler(caminhos["silver_itens"]), "id_item_pedido"))
df_rastr_s = com_lote(sem_duplicatas(ler(caminhos["silver_rastreamento"]), "id_rastreamento"))
df_cat_s = (ler(caminhos["silver_produtos"]).filter("preco_lista > 0")
            .groupBy("sku").agg(F.expr("percentile_approx(preco_lista, 0.5)").alias("preco_lista_sku")))

itens = df_itens_s.select("id_item_pedido", "id_pedido", "sku", "quantidade", "preco_unitario",
                          "desconto_aplicado", "lote", "data_lote", *REGRAS_ITENS).toPandas()
rastr = (df_rastr_s.withColumn("status_norm", status_normalizado(F.col("status_entrega")))
         .select("id_rastreamento", "id_pedido_ecommerce", "status_entrega", "status_norm", "dt_evento",
                 "id_transportadora", "nome_transportadora", "lote", "data_lote").toPandas())
catalogo = df_cat_s.toPandas()

print(f"Itens: {len(itens):,} | Pedidos: {itens.id_pedido.nunique():,} | SKUs: {itens.sku.nunique():,}")
print(f"Eventos de rastreamento: {len(rastr):,} | Pedidos rastreados: {rastr.id_pedido_ecommerce.nunique():,}")
print(f"Período dos lotes: {itens.data_lote.min()} a {itens.data_lote.max()} ({itens.lote.nunique()} lotes)")

### Etapa 2 — Adequação: nulos, tipos e duplicatas

O Isolation Forest do scikit-learn não aceita nulos. A lista de nulos por coluna já mostra o que precisa de tratamento no `09`.

In [0]:
def resumo(df, nome):
    r = pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum(),
                      "%_nulos": (df.isna().mean() * 100).round(2), "distintos": df.nunique()})
    print(f"\n=== {nome}: {len(df):,} linhas ===")
    display(r)

resumo(itens.drop(columns=REGRAS_ITENS), "itens_pedido")
resumo(rastr, "rastreamento_entregas")

print("Duplicatas de id_item_pedido após a deduplicação:", itens.id_item_pedido.duplicated().sum())
print("Itens sem pedido rastreado:", (~itens.id_pedido.isin(rastr.id_pedido_ecommerce)).sum())

### Etapa 3 — Rótulo fraco

Um item recebe `rotulo_fraco = 1` quando falha em **pelo menos uma** regra por linha da etapa 1. As regras não avaliadas (`null`) contam como aprovadas. O rótulo **não entra no treino**: serve só para medir Precision, Recall, F1 e PR-AUC do Isolation Forest.

**Referência (SQL em 05/11):** 1.234 itens com falha (8,7%). As regras 7 e 10 são agregadas por pedido e ficam de fora do rótulo por item.

In [0]:
falhas = (itens[REGRAS_ITENS].fillna(True) == False)
itens["qtd_regras_falhas"] = falhas.sum(axis=1)
itens["rotulo_fraco"] = (itens["qtd_regras_falhas"] > 0).astype(int)

por_regra = (falhas.sum().rename("itens_com_falha").to_frame()
             .assign(pct=lambda d: (d.itens_com_falha / len(itens) * 100).round(2))
             .sort_values("itens_com_falha", ascending=False))
display(por_regra)
print(f"Rótulo fraco: {itens.rotulo_fraco.sum():,} de {len(itens):,} itens ({itens.rotulo_fraco.mean():.2%})")
print("\nQuantas regras cada item reprovado quebra:")
display(itens.loc[itens.rotulo_fraco == 1, "qtd_regras_falhas"].value_counts().sort_index())

# Regras que marcam exatamente os mesmos itens são redundantes para o rótulo
r6, r9 = falhas["negocio_6_desconto_nao_maior_preco"], falhas["negocio_9_desconto_dentro_limite"]
print(f"\nR6 e R9 marcam os mesmos itens? {(r6 == r9).all()} (R6={r6.sum()}, R9={r9.sum()})")

### Etapa 4 — Distribuições de preço, quantidade e desconto

Para preço e valor, use a escala log: a mediana fica perto de R$ 20, e a cauda chega a milhares.

In [0]:
itens["razao_desconto"] = itens.desconto_aplicado / itens.preco_unitario.abs().clip(lower=0.01)
itens["valor_liquido_item"] = (itens.preco_unitario - itens.desconto_aplicado) * itens.quantidade

display(itens[["preco_unitario", "quantidade", "desconto_aplicado", "razao_desconto", "valor_liquido_item"]]
        .describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.round(2))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
pos = itens.loc[itens.preco_unitario > 0, "preco_unitario"]
ax[0].hist(np.log10(pos), bins=60, color=AZUL)
ax[0].set_title("Preço unitário (log10, só > 0)")
ax[0].set_xlabel("log10(R$)")
itens.quantidade.value_counts().sort_index().plot.bar(ax=ax[1], color=AZUL)
ax[1].set_title("Quantidade")
ax[2].hist(itens.razao_desconto.clip(-0.5, 3), bins=70, color=AZUL)
ax[2].set_title("Desconto ÷ preço (cortado em -0,5 e 3)")
plt.tight_layout()
plt.show()

faixas = pd.cut(itens.razao_desconto, [-np.inf, -1e-9, 1e-9, 0.10, 0.30, 0.50, 1.0, np.inf],
                labels=["negativo", "0", "0–10%", "10–30%", "30–50%", "50–100%", ">100%"])
print("Faixas de desconto ÷ preço:")
display(faixas.value_counts().reindex(faixas.cat.categories).to_frame("itens"))

print("Preços ≤ 0:", (itens.preco_unitario <= 0).sum(), "| Quantidade 0:", (itens.quantidade == 0).sum())

#### Valores fora do padrão que nenhuma regra marca

Este é o ponto central do P1: o modelo deve achar o que as regras fixas **não** pegam. Abaixo estão os itens **aprovados** em todas as regras, mas com preço no topo da distribuição, e as quantidades atípicas.

In [0]:
p99 = itens.preco_unitario.quantile(0.99)
fora = itens[(itens.rotulo_fraco == 0) & (itens.preco_unitario > p99)]
print(f"Itens aprovados com preço acima do p99 (R$ {p99:,.2f}): {len(fora)}")
display(fora.sort_values("preco_unitario", ascending=False)
        [["id_item_pedido", "id_pedido", "sku", "quantidade", "preco_unitario", "desconto_aplicado"]].head(15))

print("Quantidades entre os itens aprovados:")
display(itens.loc[itens.rotulo_fraco == 0, "quantidade"].value_counts().sort_index().to_frame("itens"))

#### Preço praticado × preço de catálogo

Primeiro teste de uma feature de contexto: o `preco_unitario` dividido pelo `preco_lista` mediano do SKU (Silver de produtos). Se a razão variar muito mesmo entre os itens bons, o catálogo é ruído e a feature fica de fora.

In [0]:
itens = itens.merge(catalogo, on="sku", how="left")
itens["razao_preco_catalogo"] = itens.preco_unitario / itens.preco_lista_sku
print(f"Itens sem preço de catálogo: {itens.preco_lista_sku.isna().sum()}")
display(itens.groupby("rotulo_fraco").razao_preco_catalogo
        .describe(percentiles=[.05, .25, .5, .75, .95]).round(3))

### Etapa 5 — Nível pedido

O número de itens e o valor do pedido viram features de contexto: um item "normal" pode ser estranho dentro do seu pedido.

In [0]:
pedido = itens.groupby("id_pedido").agg(n_itens=("id_item_pedido", "count"),
                                        n_skus=("sku", "nunique"),
                                        qtd_total=("quantidade", "sum"),
                                        valor_liquido=("valor_liquido_item", "sum"),
                                        tem_item_rotulado=("rotulo_fraco", "max"))
display(pedido.describe(percentiles=[.5, .9, .99]).T.round(2))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
pedido.n_itens.value_counts().sort_index().plot.bar(ax=ax[0], color=AZUL)
ax[0].set_title("Itens por pedido")
taxa = pedido.groupby(pd.cut(pedido.n_itens, [0, 2, 5, 10, 20, 100])).tem_item_rotulado.mean() * 100
taxa.plot.bar(ax=ax[1], color=AZUL)
ax[1].set_title("% de pedidos com algum item rotulado, por nº de itens")
plt.tight_layout()
plt.show()

### Etapa 6 — Rastreamento

Verifica se há dados suficientes para "horas entre eventos": quantos pedidos têm 2 ou mais eventos, quanto tempo leva cada transição e se aparecem regressões de status (por exemplo, `em transito → coletado`).

In [0]:
print("Status como chegam × normalizados:")
display(rastr.groupby(["status_norm", "status_entrega"]).size().to_frame("eventos"))

ev = rastr.groupby("id_pedido_ecommerce").size()
print("Eventos por pedido:")
display(ev.value_counts().sort_index().to_frame("pedidos"))
print(f"Pedidos com 1 evento só: {(ev == 1).mean():.1%}  →  'horas entre eventos' fica vazia nesses pedidos")

r = rastr.sort_values(["id_pedido_ecommerce", "dt_evento", "id_rastreamento"]).copy()
r["ordem"] = r.status_norm.map(ORDEM_STATUS)
g = r.groupby("id_pedido_ecommerce")
r["status_anterior"] = g.status_norm.shift()
r["ordem_anterior"] = g.ordem.shift()
r["horas_desde_anterior"] = (r.dt_evento - g.dt_evento.shift()).dt.total_seconds() / 3600
r["regressao"] = r.ordem < r.ordem_anterior

trans = (r.dropna(subset=["status_anterior"])
         .groupby(["status_anterior", "status_norm"]).horas_desde_anterior
         .agg(["count", "mean", "min", "max"]).round(1).sort_values("count", ascending=False))
display(trans)
print(f"Eventos com regressão de status: {r.regressao.sum()} | Pedidos com 'extraviado': "
      f"{r.loc[r.status_norm == 'extraviado', 'id_pedido_ecommerce'].nunique()}")

display(rastr.groupby("nome_transportadora").size().sort_values(ascending=False).to_frame("eventos"))

### Etapa 7 — As features candidatas separam o rótulo?

Para cada candidata, compara a mediana entre os itens rotulados e os não rotulados e calcula o AUC da feature sozinha. Um AUC perto de 0,5 indica pouco sinal; perto de 0 ou de 1, sinal forte. Isso não decide sozinho: o Isolation Forest procura isolamento, e não só a separação do rótulo.

In [0]:
from sklearn.metrics import roc_auc_score

rastr_ped = (r.groupby("id_pedido_ecommerce")
             .agg(n_eventos=("id_rastreamento", "count"),
                  max_horas_entre_eventos=("horas_desde_anterior", "max"),
                  qtd_regressoes_status=("regressao", "sum"))
             .reset_index().rename(columns={"id_pedido_ecommerce": "id_pedido"}))
base = (itens.merge(pedido[["n_itens", "valor_liquido"]].rename(columns={"valor_liquido": "valor_liquido_pedido"}),
                    left_on="id_pedido", right_index=True)
             .merge(rastr_ped, on="id_pedido", how="left"))
base["participacao_item"] = base.valor_liquido_item / base.valor_liquido_pedido.abs().clip(lower=0.01)

CANDIDATAS = ["preco_unitario", "quantidade", "desconto_aplicado", "razao_desconto", "valor_liquido_item",
              "razao_preco_catalogo", "n_itens", "valor_liquido_pedido", "participacao_item",
              "n_eventos", "max_horas_entre_eventos", "qtd_regressoes_status"]
linhas = []
for c in CANDIDATAS:
    x = base[[c, "rotulo_fraco"]].dropna()
    auc = roc_auc_score(x.rotulo_fraco, x[c]) if x.rotulo_fraco.nunique() == 2 else np.nan
    linhas.append({"feature": c, "mediana_rotulado": x.loc[x.rotulo_fraco == 1, c].median(),
                   "mediana_ok": x.loc[x.rotulo_fraco == 0, c].median(),
                   "AUC_isolada": auc, "forca": abs(auc - 0.5) * 2, "%_nulos": base[c].isna().mean() * 100})
display(pd.DataFrame(linhas).sort_values("forca", ascending=False).round(3))

print("Correlação de Spearman entre as candidatas:")
display(base[CANDIDATAS].corr(method="spearman").round(2))

### Etapa 8 — Estabilidade entre lotes (define o treino × teste)

O modelo vai treinar na Silver acumulada e pontuar cada micro-lote novo. Se a taxa de anomalia e os preços forem estáveis por dia, a divisão **temporal** (dias mais antigos no treino, os mais recentes no teste) é justa.

In [0]:
dia = itens.groupby("data_lote").agg(itens=("id_item_pedido", "count"),
                                     pct_rotulado=("rotulo_fraco", "mean"),
                                     preco_mediano=("preco_unitario", "median"),
                                     pct_com_desconto=("desconto_aplicado", lambda s: (s > 0).mean()))
display(dia.assign(pct_rotulado=lambda d: (d.pct_rotulado * 100).round(2),
                   pct_com_desconto=lambda d: (d.pct_com_desconto * 100).round(2)))

fig, ax = plt.subplots()
ax.bar(dia.index.astype(str), dia.pct_rotulado * 100, color=AZUL)
ax.set_title("% de itens com rótulo fraco por dia de lote")
ax.set_ylabel("%")
plt.show()

acum = dia.itens.cumsum() / dia.itens.sum()
print("Proporção acumulada de itens por dia (para escolher o corte de treino):")
display(acum.round(3).to_frame("acumulado"))

### Etapa 9 — Conclusões e insights

> Resultados da execução de 07/11 (Silver com lotes de 21/09 a 15/10).

**Adequação dos dados**
- 14.196 itens em 2.375 pedidos e 3.075 SKUs, com 7 dias de lote. Não há nulos em preço, desconto ou quantidade. A base é adequada para o Isolation Forest.
- **Rótulo fraco: 1.234 itens (8,69%)**, igual à Gold. Entre os reprovados, 548 quebram 1 regra, 551 quebram 2, 123 quebram 3 e 12 quebram 4 ou mais. A taxa do rótulo orienta o `contamination` do modelo (perto de 0,09).
- Todas as falhas por item vêm de regras **numéricas**: as regras de FK não reprovaram nenhum item.
- **A base é estável por dia:** o % rotulado fica entre 8,3% e 10,1%, o preço mediano entre R$ 17,60 e R$ 22,75, e cerca de 18% dos itens têm desconto. Por isso, a divisão temporal é justa.

**Poder de separação das features (Etapa 7)**
| Feature | AUC isolada | Força | Leitura |
|---|---|---|---|
| `valor_liquido_item` | 0,000 | 1,00 | Separa o rótulo quase perfeitamente: o item reprovado tem valor ≤ 0 (mediana −0,94 contra R$ 25,28) |
| `participacao_item` | 0,000 | 1,00 | Herda o mesmo efeito do valor líquido |
| `razao_desconto` | 0,712 | 0,42 | Forte, sem redundância com o valor |
| `quantidade` | 0,304 | 0,39 | Forte (quantidade 0) |
| `preco_unitario` | 0,414 | 0,17 | Moderada |
| `razao_preco_catalogo` | 0,420 | 0,16 | Fraca, e anda junto com o preço (Spearman 0,73), por isso **fica fora** |
| `valor_liquido_pedido`, `n_itens` | 0,42 / 0,51 | 0,16 / 0,01 | Contexto, com pouco sinal sozinhas |
| Rastreamento (`max_horas`, `n_eventos`, `regressoes`) | ≈ 0,5 | ≤ 0,13 | **Nenhum sinal contra o rótulo**, e 90% de nulos em `max_horas` |

**Insights**
1. **Valor líquido ≤ 0 resume as regras numéricas:** quantidade 0, preço ≤ 0 e desconto maior que o preço levam o valor do item para zero ou abaixo. A feature é legítima, porque vem dos dados brutos e está no plano, mas deixa a comparação com o rótulo fácil demais. O modelo será avaliado **com e sem** `log_valor_liquido_item` e `participacao_item`, para medir o que ele encontra sem essa "pista".
2. **Desconto em dois grupos:** cerca de 82% dos itens não têm desconto, cerca de 14% têm até 30%, e 679 itens têm desconto acima de 100% do preço. Não há nada entre 30% e 100%, então as **regras 6 e 9 marcam os mesmos 679 itens**. Uma das duas pode ser revista na etapa 1.
3. **Valores fora do padrão sem regra:** preços de R$ 674,50, R$ 1.072 e R$ 5.292 (mediana de R$ 20, p99 de R$ 299,99), além das quantidades 6 e 12, que podem ser kits. São os candidatos a "anomalia que a regra fixa não pega".
4. **O rastreamento não explica anomalias de item:** 89% dos pedidos têm 1 evento só, e as horas entre transições são regulares. As regressões de status e o extravio são anomalias **de entrega**, e não de item.
5. **Divisão temporal:** o dia 08/10 sozinho tem 5.250 itens (37% da base). Com o corte em 07/10, o treino tem 5 dias e 8.075 itens (8,95% rotulados), e o teste tem 2 dias e 6.121 itens (8,35% rotulados), ou seja, 57% / 43%. Colocar 08/10 no treino deixaria só 871 itens no teste.

**Decisões levadas ao `09` e ao `10`**
- As colunas booleanas das regras ficam fora das features, porque dariam o próprio rótulo ao modelo.
- Preço e valores entram com log com sinal.
- Rastreamento: as features são gravadas na tabela, mas a **versão principal do modelo usa só item e pedido (9 features)**. O rastreamento entra numa versão comparativa, porque features sem sinal diluem os cortes aleatórios do Isolation Forest.